# Grok Register · Colab 无代理版

**说明**
- 不修改仓库业务代码；只用 `colab/run_colab_register.py` 在本进程打补丁。
- **不用代理**，出口 = 当前 Colab 机器 IP（多为 Google 机房，可能更容易 bot_flag）。
- **不能**在断线后自动换机继续跑；换机 = 断开 Runtime → 重连（新 IP）→ 再跑本笔记本。

**你要做的**
1. 从上到下运行每个单元格  
2. 准备好 `config.json`（邮箱等）  
3. 跑完下载输出文件

## 1. 安装系统依赖 + Python 包
首次运行较慢，可喝口水。

In [ ]:
%%bash
set -e
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y -qq chromium-browser chromium-chromedriver fonts-liberation \
  libnss3 libatk-bridge2.0-0 libgtk-3-0 libx11-xcb1 libxcomposite1 libxdamage1 \
  libxrandr2 libgbm1 libasound2t64 libpangocairo-1.0-0 libcups2 >/dev/null 2>&1 \
  || apt-get install -y -qq chromium-browser chromium-chromedriver fonts-liberation \
  libnss3 libatk-bridge2.0-0 libgtk-3-0 libx11-xcb1 libxcomposite1 libxdamage1 \
  libxrandr2 libgbm1 libasound2 libpangocairo-1.0-0 libcups2

python -m pip install -q -U pip
python -m pip install -q DrissionPage curl_cffi requests psutil lxml cssselect \
  websocket-client tldextract filelock DataRecorder DownloadKit

which chromium-browser || which chromium || true
chromium-browser --version 2>/dev/null || chromium --version 2>/dev/null || true
echo "deps ok"

## 2. 获取源码

**二选一**：
- A：从 GitHub clone（改成你的仓库地址）
- B：左侧文件栏上传整个项目 zip 后解压

下面默认 A。

In [ ]:
import os
from pathlib import Path

# ====== 按需修改 ======
REPO_URL = "https://github.com/AaronL725/grok-register.git"  # 或你的 fork
BRANCH = "main"
WORK = Path("/content/grok-register")
# =====================

if not (WORK / "grok_register_ttk.py").exists():
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {WORK}
else:
    print("repo already present:", WORK)

os.chdir(WORK)
print("cwd =", os.getcwd())
assert Path("colab/run_colab_register.py").is_file(), "缺少 colab/run_colab_register.py，请确认分支含 colab 目录"

## 3. 放入 config.json

任选一种：
1. 左侧上传 `config.json` 到 `/content/grok-register/config.json`
2. 或在下一格粘贴 JSON 覆盖写入

至少要能注册：邮箱服务商密钥、域名等。

In [ ]:
from pathlib import Path
import json

cfg_path = Path("/content/grok-register/config.json")
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    print("loaded config.json, keys=", len(cfg))
    print("email_provider=", cfg.get("email_provider"))
    print("chenyme_enabled=", cfg.get("chenyme_grok2api_enabled"))
    print("cpa_export_enabled=", cfg.get("cpa_export_enabled"))
else:
    print("尚未找到 config.json — 请上传到", cfg_path)
    print("或把完整 JSON 赋给 CONFIG_JSON 变量后取消注释写入逻辑")

# CONFIG_JSON = r'''{ ... 你的 config ... }'''
# cfg_path.write_text(CONFIG_JSON, encoding="utf-8")

## 4. 看当前出口 IP

In [ ]:
import json
from curl_cffi import requests

r = requests.get("https://ipinfo.io/json", timeout=15, proxies={})
info = r.json()
print(json.dumps(info, indent=2, ensure_ascii=False))
org = str(info.get("org") or "").lower()
hosting = any(x in org for x in ("google", "cloud", "amazon", "microsoft", "hosting", "server"))
print("hosting_like=", hosting)
if hosting:
    print("提示: 机房 IP 跑 Build 更容易 bot_flag；可 Runtime→Disconnect and delete runtime 换机后再跑")

## 5. 开始注册

改 `COUNT`。默认关 chenyme convert；需要本地 CPA 时加 `--enable-cpa`。

In [ ]:
import os
os.chdir("/content/grok-register")

COUNT = 2  # 本机本次注册几个

# 无代理注册；默认关远程 convert
!python colab/run_colab_register.py --count {COUNT}

# 若希望每 1 个号就请求断开 Runtime 换机（断后需手动重连再跑本格）：
# !python colab/run_colab_register.py --count 1 --rotate-after 1

## 6. 打包下载结果

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

root = Path("/content/grok-register")
out_dir = Path("/content/colab_out")
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir()

patterns = [
    "accounts_*.txt",
    "grok2api_build_import.json",
    "liveness_*.jsonl",
    "reprobe_*.jsonl",
]
for pat in patterns:
    for p in root.glob(pat):
        shutil.copy2(p, out_dir / p.name)
        print("+", p.name)

cpa = root / "cpa_auths"
if cpa.is_dir():
    shutil.copytree(cpa, out_dir / "cpa_auths")
    print("+ cpa_auths/")

zip_path = "/content/grok_register_colab_out"
shutil.make_archive(zip_path, "zip", out_dir)
files.download(zip_path + ".zip")
print("downloaded", zip_path + ".zip")

## 7. 手动换机（新 IP）

菜单：**Runtime → Disconnect and delete runtime** → 重新连接 → **Runtime → Run all**。

或运行下一格（会断开当前会话）：

In [ ]:
# 危险：会断开当前 Colab 会话
DO_UNASSIGN = False  # 改 True 再运行

if DO_UNASSIGN:
    from google.colab import runtime
    print("unassign… 请稍后重新连接")
    runtime.unassign()
else:
    print("未执行。把 DO_UNASSIGN=True 再跑，或用菜单 Disconnect and delete runtime")